# Глава 10. Альтернативные архитектуры

Эта глава — обзор генеративных языковых моделей, которые в той или иной степени отходят от стандартного Transformer с полным self-attention

Transformer победил рекуррентные сети по одной практической причине: self-attention перемешивает всю последовательность за один параллельный шаг, и поэтому обучение прекрасно ложится на GPU. Но у этого механизма есть  цена. Внимание сравнивает каждый токен с каждым, поэтому вычисления и память на шаге attention растут как O(N²) по длине последовательности N. Удвоение длины контекста учетверяет стоимость, и на контекстах в сотни тысяч токенов это становится запретительно дорого.

Вторая проблема проявляется не при обучении, а при инференсе. Авторегрессионный Transformer на каждом новом токене обращается ко всем предыдущим, и чтобы не пересчитывать их заново, он хранит так называемый KV-кэш — запомненные ключи и значения для всех прошлых позиций. Этот кэш растёт линейно с длиной диалога и съедает память тем сильнее, чем длиннее контекст.

Рекуррентные сети (RNN, LSTM) генерировали токены по одному, держа фиксированное по размеру скрытое состояние, поэтому их инференс был дёшев и по памяти постоянен. Но обучались они последовательно и плохо масштабировались. Transformer выбрал противоположную точку: дорогой по памяти, но идеально параллелизуемый. Почти все рассматриваемые ниже архитектуры пытаются получить лучшее из двух миров — параллельное обучение, как у Transformer, и дешёвый рекуррентный инференс с постоянной памятью, как у RNN


### Transformer-XL
Классический Transformer не может обрабатывать тексты длины большей, чем заявленный лимит (для оригинальной модели BERT это, например, всего 512 токенов). Поэтому если текст длинный, его резали на сегменты, и модель работала с каждым куском независимо. Для таких кейсов [(Dai et al, 2019)](https://arxiv.org/abs/1901.02860) попробовали создать гибрид трансофрмера и старой доброй рекуррентной сети и связать несколько итераций применения трансформера в одну цепочку. В рамках этой модели трансформер вычисляет ответ для каждого сегмента текста и передает свои скрытые состояния (от всех слоев) на следующую итерацию до тех пор пока не будет обработан весь текст. Так становится возможным работать с очень большими контекстами: последний сегмент текста считается Трансформером, а сигналы от старых сегментов ужимаются в одно скрытое состояние и агрегируются с текущими.

Актуально и для обучения, и для инференса. При обучении обновляются веса на базе текущего сегмента, вход старых сегментов при этом замораживают.

<img src="img/transformer_xl.png" width=500>

Чтобы это заработало, понадобилось переизобрести позиционное кодирование. Абсолютные позиции при склейке сегментов начинают конфликтовать, поэтому авторы ввели относительное позиционное кодирование, где внимание зависит от расстояния между токенами, а не от их абсолютных индексов. Эта идея относительных позиций пережила саму модель и встречается во множестве более поздних архитектур

### Linear Transformers
[(Katharopoulos et al, 2020)](https://arxiv.org/abs/2006.16236) <br>Напомним как выглядит вычисление стандартного attention:
$$\text{softmax}\left(\frac{QK^T}{\sqrt{D}}\right) V = 
\sum_{j=1}^N \left[ \frac{\exp\left(\frac{Q_i K_j^T}{\sqrt{D}}\right)}{\sum_{j=1}^N \exp\left(\frac{Q_i K_j^T}{\sqrt{D}}\right)} \right] \cdot V_j = 
\frac{\sum_{j=1}^N \exp\left(\frac{Q_i K_j^T}{\sqrt{D}}\right) V_j}{\sum_{j=1}^N \exp\left(\frac{Q_i K_j^T}{\sqrt{D}}\right)}
$$

Давайте перепишем его в обобщенном виде. Можем обратить внимание, что $QK^T$ выполняет роль некоторой функции близости между Q и K

$$O_i = \frac{\sum_{j=1}^N \text{sim}(Q_i, K_j) V_j}{\sum_{j=1}^N \text{sim}(Q_i, K_j)}$$

Тогда формулировка с softmax-ом будет просто частный случай, где $\text{sim}(Q_i, K_j) = \exp{(QK^T)}$. А что еще хорошо моделирует близость - ядровая функция

---

В математике ядро $K(x,y)$ - это любая функция двух переменных, которую можно представить в виде скалярного произведения двух отображений каждой переменной $K(x,y) = \phi(x) \cdot \phi(y)$. То есть как сумму произведений фичей X и Y<br><img src="img/kernel.png" width=350><br>Среди известных ядер выделяют 4: тривиальное, полиномиальное, экспоненциальное, еще<br>

---

Так почему бы вместо экспоненты во внимании не использовать какое-либо ядро $\text{sim}(Q_i, K_j) = \phi(Q_i)^T \phi(K_j)$. Тем более, что эта экспонента уже очень похожа на экспоненциальное RBF ядро. Это открывет возможность позволило бы распутать нелинейную зависимость QK^T и представить Attention как последовательное произведение матриц. 

$$O_i = \frac{\sum_{j=1}^N \phi(Q_i)^T \phi(K_j) V_j}{\sum_{j=1}^N \phi(Q_i)^T \phi(K_j)}$$

Константные слагаемые, относящиеся к вектору запроса, можно вынести за скобку:

$$O_i = \frac{\phi(Q_i)^T \sum_{j=1}^N \phi(K_j) V_j}{\phi(Q_i)^T \sum_{j=1}^N \phi(K_j)}$$

Теперь суммы в числителе и знаменателе можно не пересчитывать, а накапливать. Обзовем их S и Z<br>
$S_i = S_{i-1} + \phi(K_i) V_i^T$<br>
$Z_i = Z_{i-1} + \phi(K_i)$

То есть выход для токена $X_i$ считается по выходам предыдущих токенов, появляется аналогия с RNN.
$$O_i = \frac{\phi(Q_i)^T S_i}{\phi(Q_i)^T Z_i}$$

Это в первую очередь полезно для инференса, там вычисление для каждого нового токена становится константное, а общая сложность генерации O(N)

Для обучении вместо вычисления связей для каждого токена, числитель считается независимо (параллельно), а знаменатель суммируется вторым проходом. Итого сложность $O(2N)$ вместо O(N^2).

### RWKV
[(Peng et al, 2023)](https://arxiv.org/abs/2305.13048) <br>Метод опирается на идею Attention Free Transformer ([Zhai et al, 2021](https://arxiv.org/abs/2105.14103)), предложенную двумя годами ранее в Apple. Там внимание считали по альтернативной формуле:
$$Y_t = \sigma_q(Q_t) \odot \frac{\sum_{t'=1}^{T} \exp(K_{t'} + w_{t,t'}) \odot V_{t'}}{\sum_{t'=1}^{T} \exp(K_{t'} + w_{t,t'})}$$

В RWKV считается только текущая связь KV, прибавляется к накопленному вектору S

Схема всего этого:<br>
<img src="img/rwkv.png" width=350><img src="img/rwkv2.png" width=350>

Архитектура RMWKV блока:
- Слой Time Mixing (аналог Attention)<br><br>
    - Шаг Token-shift: смесь текущего входа с предыдущим. На первом слое с эмбедингом, на последующих с промежуточным выходом $$\text{shift}(x_t, \mu) = \mu \odot x_t + (1 - \mu) \odot x_{t-1}$$Зачем - поскольку рекурркнтная схема вычисления никакого явного порядка в накопленном сигнале не подразумевает, token shift позволяет сориентировать модель, в каком месте генерации она находится. Щепотка внимания в классический RNN подход<br><br>
    - Шаг WKV считает взвешенную сумму всех предыдущих Value векторов. Вес определяется вектором Key: $$wkv_t = \frac{\sum_{i=1}^{t-1} e^{k_i} \odot v_i}{\sum_{i=1}^{t-1} e^{k_i}}$$ Добавляется поправка на затухание $w$, это обучаемый вектор той же размерности что Key: $$wkv_t = \frac{\sum_{i=1}^{t-1} e^{-(t-1-i)w + k_i} \odot v_i}{\sum_{i=1}^{t-1} e^{-(t-1-i)w + k_i}}$$ Добавляют еще "бонусный" вектор: $$wkv_t = \frac{\sum_{i=1}^{t-1} e^{-(t-1-i)w + k_i} \odot v_i + e^{u + k_t} \odot v_t}{\sum_{i=1}^{t-1} e^{-(t-1-i)w + k_i} + e^{u + k_t}}$$<br><br>
    - Шаг Reception это просто гейтинг для выбора, какая часть сигнала подйет дальше в зависимости от обрабатываемого сейчас токена $$o_t^{time} = W_o \cdot (\sigma(r_t) \odot wkv_t)$$<br>
- Слой Channel Mixing (аналог FFN)<br>
    - Token Shift
    - Mixing $$o_t^{channel} = \sigma(r_t') \odot (W_v' \cdot \text{ReLU}(k_t')^2)$$



RWKV прошёл несколько поколений (версии Eagle и Finch ввели зависящую от данных рекуррентность, концептуально близкую к селективности Mamba) и вырос до моделей масштаба около 14 миллиардов параметров, конкурентных с трансформерами схожего размера. Отдельная ценность RWKV — дешёвый инференс на слабом железе.

### RetNet 
[(Sun et al, 2023)](https://arxiv.org/abs/2307.08621)<br>
Retentive Network, заходит с той же целью, но формулирует механизм retention, который явно выводит связь между рекуррентностью и вниманием. Его «фишка» — три эквивалентных режима вычисления одной и той же модели: параллельный (для быстрого обучения), рекуррентный (для дешёвого пошагового инференса) и chunkwise-рекуррентный (компромисс, когда последовательность бьётся на блоки, внутри блока считается параллельно, а между блоками — рекуррентно). Эта триада «параллельное обучение плюс рекуррентный инференс» — общая мечта всего направления, и RetNet формулирует её особенно чисто.


## State Space Models
Параллельно линеаризации внимания вырос совсем другой по происхождению класс моделей — модели пространства состояний (state space models, SSM), идущие из теории управления и обработки сигналов.

Потрясающее направление, столько математики нет ни в одном другом инженерном LLM проекте. Тут и теория управления, и дифференциальные уравнения, и преобразование сигналов, и теория приближений, и комплексный анализ, и численные методы.

Structured State Space. SSM описывает последовательность через непрерывную линейную динамическую систему: есть скрытое состояние, которое эволюционирует под действием входа по фиксированным матрицам. У такой системы есть замечательное свойство: её можно представить и как рекуррентность (удобно для инференса), и как одну длинную свёртку (удобно для параллельного обучения). Грамотная структурная параметризация матриц позволила S4 эффективно ловить очень длинные зависимости — там, где attention захлёбывался по памяти.

### S4 (Gu и соавторы, 2021)
Авторы вдохновляются теорией управления (control theory) и заимствуют оттуда уравнения, описывающие процесс как линейную динамическую систему с непрерывным временем $t$:

$$
\begin{aligned}
\mathbf{x}'(t) &= \mathbf{A}\,\mathbf{x}(t) + \mathbf{B}\,\mathbf{u}(t) \quad &\text{(уравнение состояния)} \\
\mathbf{y}(t) &= \mathbf{C}\,\mathbf{x}(t) + \mathbf{D}\,\mathbf{u}(t) \quad &\text{(уравнение выхода)}
\end{aligned}
$$

где x(t) - вектор состояния системы, u(t) - вектор управления, y(t) - выход системы, A - матрица смены состояния, C - вектор генерации выхода, B матрицы входа, D - (опциональный) скаляр

Обратите внимание, что поскольку процесс непрерывный, единственный способ его линейно описать - через производную ($\dot{x}$). Рекуррентное соотношение не существует. Чтобы его получить, нужно перейти к дискретной постановке

Дискретизация подразумевает переход от непрерывного сигнала $x(t)$ к набору наблюдений $x_k$, такой сигнал уже можно скормить вычислителю. Есть разные подходы, приведем три:
- __метод Эйлера__<br>апроксимируем производную конечной разностью $\frac{(x(t+\Delta) - x(t))}{\Delta}$ и выражаем $x(t+\Delta)$ через матрицы<br><img src="img/euler.png" width=200><br><br>
- [__zero order hold__](https://en.wikipedia.org/wiki/Zero-order_hold)<br>если мы считаем, что u(t) кусоно-постоянный сигнал, то можно прямо получить аналитическое решение, решая дифференциальное уравнение, однако интегрирование трудоемко<br><br>
- [__метод Тастина__](https://en.wikipedia.org/wiki/Bilinear_transform)<br>интеграл апроксимируется по методу трапеций и мы получаем простое выражение для $\overline{A}$ и $\overline{B}$. Важно, что такое приближение сохраняет устойчивость, A не сжимает и не взрывает состояние<br>$$
\overline{\mathbf{A}} = \left(\mathbf{I} - \frac{\Delta}{2}\mathbf{A}\right)^{-1}\left(\mathbf{I} + \frac{\Delta}{2}\mathbf{A}\right), \quad
\overline{\mathbf{B}} = \left(\mathbf{I} - \frac{\Delta}{2}\mathbf{A}\right)^{-1} \Delta \mathbf{B}.
$$

Теперь у нас не проризводная, а рекурсия x = f(x)

$$
\begin{aligned}
\mathbf{x_{t+1}} &= \mathbf{\overline{A}}\,\mathbf{x_t} + \mathbf{\overline{B}}\,\mathbf{u_t} \quad &\text{(уравнение состояния)} \\
\mathbf{y_{t+1}} &= \mathbf{C}\,\mathbf{x_t} \quad &\text{(уравнение выхода)}
\end{aligned}
$$

Теперь видна аналогия с RNN: состояние обновляется линейно, выход — линейная проекция. Параметр $\Delta$ управляет «временным разрешением». Если $\Delta$ мало, система обновляется быстро; если велико — медленно, запоминая дальний контекст.

Рекуррентную формулу надо развернуть

$$
\begin{aligned}
\mathbf{x}_0 &= \overline{\mathbf{B}} u_0 \\
\mathbf{x}_1 &= \overline{\mathbf{A}}\,\overline{\mathbf{B}} u_0 + \overline{\mathbf{B}} u_1 \\
\mathbf{x}_2 &= (\overline{\mathbf{A}})^2\overline{\mathbf{B}} u_0 + \overline{\mathbf{A}}\,\overline{\mathbf{B}} u_1 + \overline{\mathbf{B}} u_2 \\
&\vdots \\
\mathbf{x}_k &= \sum_{j=0}^{k} (\overline{\mathbf{A}})^{k-j} \, \overline{\mathbf{B}} \, u_j.
\end{aligned}
$$

А выход $y_k$ будет соотвественно:
$$
y_k = \mathbf{C}\mathbf{x}_k = \sum_{j=0}^{k} \left(\mathbf{C}
\,\,(\overline{\mathbf{A}})^{k-j}\,\,\overline{\mathbf{B}}\right) u_j.
$$

Можно заметить, что данное выражение в точности соотвествует определению свертки $
\mathbf{y} = \overline{K} * \mathbf{u}
$, где набор степенных матриц $\left[\mathbf{C}, \mathbf{C}\mathbf{\overline{A}}, C\overline{\mathbf{A}}\overline{\mathbf{B}} ... С\mathbf{\overline{A}}^{k-j}\,\,\overline{\mathbf{B}}\right]$ является ядром $\overline{K}$ этой свертки и имеет длину $L$. 

В теории обработки сигналов есть [теорема о свертке](https://ru.wikipedia.org/wiki/%D0%A2%D0%B5%D0%BE%D1%80%D0%B5%D0%BC%D0%B0_%D0%BE_%D1%81%D0%B2%D1%91%D1%80%D1%82%D0%BA%D0%B5). Она гласит что свёртка во временно́й области эквивалентна поэлементному умножению в частотной области => вместо прямого вычисления свертки мы можем перевести векторы $x$ и $u$ в пространство частот с помощью DFT, перемножить их за O(N) и вернуть обратно для $y$

Быстрое преобразование Фурье (FFT) требует $O(L \log L)$. На этапе инференса можно переключиться на рекуррентное вычисление за $O(L)$ с константной памятью — идеальное свойство для автогенерации текста

В 2020 году был предложен фреймворк **HiPPO** (High-order Polynomial Projection Operators). Он формулирует задачу: скрытое состояние $\mathbf{x}(t)$ должно сжимать историю $u(\tau), \tau \le t$ оптимальным образом — проецируя её на ортогональные полиномы Лежандра с экспоненциально затухающей мерой. 

Утверждается что Решением этой задачи является конкретная матрица $\mathbf{A}_{\text{HiPPO}}$ размера $N \times N$ рекуррентная система запоминает историю с ошибкой, экспоненциально убывающей по $N$. Именно это свойство стало ключом к преодолению «забывания» обычных RNN.

$$
\mathbf{A}_{nk} = -\begin{cases}
\sqrt{(2n+1)(2k+1)} & \text{если } n > k \\
n+1 & \text{если } n = k \\
0 & \text{если } n < k
\end{cases}
$$

Возведение в степень или обращение плотной матрицы требует $O(N^3)$ операций. В алгебре когда хотят упростить работу с матрицей прибегают к ее разложению на диагональную + низкоранговую. В S4 делают так же, раскладывают HiPPO-матрицу в то, что они называют "Diagonal Plus One Rank" (DPLR) форму, при этом разложение остается точным:

$$
\mathbf{A} = \mathbf{\Lambda} - \mathbf{P}\mathbf{Q}^*,
$$

где $\mathbf{\Lambda} = \text{diag}(\lambda_1, \dots, \lambda_N)$ — диагональная матрица комплексных собственных чисел с отрицательной вещественной частью (устойчивость), $\mathbf{P}, \mathbf{Q} \in \mathbb{C}^{N \times 1}$ — векторы, $*$ означает сопряжённое транспонирование. 

Благодаря DPLR-структуре, свёрточное ядро $\overline{K}$ можно вычислить за $O(N \log N + L \log L)$, а не $O(N^2L)$. Ключевой трюк — использование тождества Вудбери для обращения и полиномиальной теории вычетов для вычисления $\overline{K}_j$.

На практике матрицы $\mathbf{A}, \mathbf{B}, \mathbf{C}, \Delta$ становятся обучаемыми параметрами, но $\mathbf{A}$ инициализируется из HiPPO, а затем преобразуется в DPLR-форму

### H3 
[(Fu et al, 2022)](https://arxiv.org/pdf/2212.14052)<br>
H3 = Hungry Hungry Hippos. Авторы перестроили SSM-блок так, чтобы он лучше выполнял операции, важные для языкового моделирования, например сопоставление и копирование токенов. Добавили:

- Shifted window attention<br>локальное самовнимание со скользящим окном, имеет фиксированную длину (например, 64 или 128 токенов)

- Gated SSM<br>входной сигнал расщепляется на две ветви: одна проходит через SSM блок как в S4, другая — через нелинейность (например, SiLU), затем они поэлементно перемножаются. Это добавляет нелинейность и позволяет динамически регулировать поток информации, подобно вентилям LSTM или gated linear units (GLU)

- Gated MLP
Стандартный полносвязный слой с активацией, также со стробированием (SwiGLU). Он отвечает за поканальное смешивание признаков.

### Hyena 
[(Poli et al, 2023)](https://arxiv.org/abs/2302.10866)<br>
В методе заменили внимание на иерархию длинных свёрток с управляемым по данным гейтингом, получив субквадратичную и полностью бесвниманческую (attention-free) архитектуру.

Авторы задались вопросом: можно ли вообще обойтись без явного механизма внимания? Их ответ — заменить фиксированное свёрточное ядро S4 на входозависимое неявное ядро, которое моделирует дальние взаимодействия через иерархию простых операций.

Вместо того чтобы вычислять внимание (квадратичная сложность), Hyena представляет выход как нелинейную свёртку:

$$y=H(u)∗v$$
где H(u) — я́дро, зависящее от входа, а v — линейная проекция входа (аналог value в трансформере). Само ядро формируется маленькой нейронной сетью, но главная хитрость в том, что его не нужно явно вычислять в плотной форме. Вместо этого используется иерархия матричных умножений и быстрых свёрток:

Вход проецируется в несколько «голов» (heads)

Для каждой головы ядро параметризуется как комбинация небольших фильтров, которые применяются рекурсивно: на каждом уровне иерархии сигнал умножается на обучаемую матрицу и сворачивается с коротким ядром (длиной O(logL)), что в совокупности имитирует длинное ядро.

Умножение сигнала на матрицу в каждом шаге делает ядро входозависимым (похоже на то, как attention зависит от Q и K)

Фактически Hyena — это нелинейное обобщение S4: если S4 использует фиксированное линейное ядро K, то Hyena строит ядро динамически, сохраняя эффективность через БПФ и иерархическую факторизацию. Эксперименты показали, что Hyena может конкурировать с трансформерами на задачах длины до 32K токенов, не уступая им в перплексии, но с сублинейной сложностью.

Если H3 сохраняет явное (хотя и локальное) внимание, то Hyena пытается полностью его исключить, «вшивая» его свойства в саму структуру свёрточного ядра

Кульминация направления 

### Mamba
[(Gu и Dao, 2023)(sdfsdf) <br>
У классических SSM был принципиальный недостаток: их матрицы фиксированы и не зависят от входа (линейно-инвариантны во времени), поэтому модель не умела избирательно запоминать или забывать содержимое в зависимости от контекста — а именно это хорошо делает attention. Mamba вводит selective state space: параметры SSM становятся функцией от текущего входа, и модель учится решать, что протащить в состояние, а что отбросить. Это даёт ей способность к контентно-зависимой фильтрации, раньше доступную в основном вниманию. Расплата в том, что зависящие от входа матрицы ломают трюк со свёрткой и заставляют считать рекуррентность последовательно, что плохо для GPU. Поэтому второй ключевой вклад Mamba — аппаратно-осведомлённый параллельный скан (hardware-aware parallel scan): алгоритм с объединением ядер (kernel fusion) и аккуратной работой с иерархией памяти GPU, который не материализует раздутые состояния и делает обучение практичным. На инференсе Mamba исполняется как рекуррентность — один шаг обновления состояния на токен, без растущего KV-кэша.

Год спустя появилась __Mamba-2__ [(Dao и Gu, 2024)](https://arxiv.org/abs/2405.21060). Её главный результат скорее теоретический: авторы показали, что модели пространства состояний и внимание — две стороны одной математической конструкции, и сформулировали это как state space duality (SSD). Практически это позволило переписать вычисления Mamba через матричные умножения и заметно ускорить обучение. Линия SSM продолжает развиваться и после этого, но именно дуальность «attention ↔ SSM» стала важной концептуальной вехой: граница между двумя лагерями оказалась куда более размытой, чем считалось.


## xLSTM: масштабирование LSTM

Отдельная и любопытная ветка задаёт вопрос с другого конца: а что, если не изобретать новый механизм, а взять классический LSTM из 1990-х, устранить его известные ограничения и масштабировать до миллиардов параметров современными приёмами? Так появился __xLSTM__ [(Beck и соавторы, 2024)](https://arxiv.org/abs/2405.04517)

Авторы вносят два главных изменения. Первое — экспоненциальное гейтирование (exponential gating) вместо сигмоидного, с дополнительными приёмами нормализации и стабилизации, чтобы экспоненты не уходили в переполнение. Это даёт более гибкий контроль над тем, что запоминать и забывать. Второе — пересмотр структуры памяти, и здесь вводятся два типа ячеек. sLSTM хранит скалярную память и добавляет новый механизм перемешивания памяти (memory mixing) через рекуррентные связи, оставаясь по духу последовательным. mLSTM заменяет скалярную память на матричную: состояние становится матрицей, которая хранит пары «ключ-значение» через внешнее произведение (covariance update rule), а извлечение делается матричным умножением. Принципиально, что mLSTM полностью параллелизуем при обучении и работает за линейное время на инференсе.

Из этих ячеек собирают остаточные блоки (xLSTM-блоки), которые стопкой образуют сеть. Концептуально xLSTM близок к RetNet, RWKV и линейным трансформерам — все они так или иначе крутятся вокруг идей матричной памяти и гейтинга. Главный посыл работы прост: при должном масштабировании рекуррентная по своей сути архитектура может конкурировать с трансформерами и SSM и по качеству, и по масштабируемости. Это ещё одно подтверждение сквозной темы — область упорно переоткрывает рекуррентность.


## Гибридные архитектуры

Накопив опыт, исследователи заметили устойчивую закономерность: линейные и SSM-модели дёшевы и отлично держат длинный контекст, но систематически проигрывают вниманию в задачах точного извлечения — когда нужно дословно достать конкретный факт из давно прочитанного места или строго следовать формату из примеров (in-context learning). Внимание, наоборот, дорого, но в точечном извлечении почти безупречно. Отсюда естественный инженерный ответ — гибриды.

Идея гибридной архитектуры: построить большую часть слоёв на дешёвом механизме (Mamba или линейное внимание), а в нескольких стратегических местах вставить полноценные слои attention — ровно там, где нужна точность извлечения. Так получают почти линейную стоимость в среднем, сохраняя сильные стороны внимания.

Примеров уже много. __Jamba__ [(AI21, 2024)](https://arxiv.org/abs/2403.19887) чередует слои Mamba и attention и добавляет mixture-of-experts для параметрической эффективности (порядка 52 миллиардов параметров суммарно при около 12 активных на инференсе); команда отдельно отмечала, что чистая Mamba хуже справлялась с in-context learning, что и подтолкнуло к гибриду

<img src="img/jamba.png" width=400>

__Samba__ комбинирует Mamba со скользящим окном внимания, __Codestral Mamba__ (Mistral) — это, наоборот, почти чистая Mamba-2 для генерации кода, а среди гибридов также называют __Zamba__ и IBM Granite 4.0. Линию подхватывают и крупные вендоры, размещающие большинство слоёв на SSM и точечно — внимание для задач, требующих дословного извлечения

Стоит отдельно отметить, что mixture-of-experts (MoE), часто встречающийся рядом, — это ортогональная идея. MoE меняет не механизм перемешивания последовательности, а способ масштабировать число параметров через условные вычисления (для каждого токена активируется лишь часть «экспертов»). Поэтому MoE не относится к «beyond attention» в строгом смысле и сочетается с любой из перечисленных архитектур.


## Диффузионные языковые модели
Здесь attention может оставаться нетронутым, под вопрос ставится порядок генерации - авторегрессия

Диффузионные модели произвели революцию в генерации изображений: они начинают с шума и постепенно его очищают до картинки. Перенести это на дискретный текст оказалось нетривиально, и сложилась отдельная линия работ. 

__D3PM__ [(Austin et al, 2021)](https://arxiv.org/abs/2107.03006)<br>задал общий каркас дискретной диффузии в пространстве категориальных состояний. 

__Diffusion-LM__ [(Li et al, 2022)](https://arxiv.org/pdf/2205.14217)<br>пробовал диффузию в непрерывном пространстве эмбеддингов

<img src="img/diffusion_lm.png" width=450>

__SEDD__ [(Lou et al, 2024)]()<br>уточнил вероятностную постановку для дискретной диффузии. Особенно практичной оказалась так называемая masked (absorbing) diffusion, где «зашумление» — это маскирование токенов, а обратный процесс — их предсказание.

__LLaDA__ [(Nie et al, 2025)](https://arxiv.org/abs/2502.09992)<br>LLADA = Large Language Diffusion with mAsking. довела этот подход до масштаба, сопоставимого с сильными авторегрессионными моделями. Прямой процесс случайно маскирует токены с долей, выбираемой равномерно от нуля до единицы; обратный процесс — это предсказатель масок, который параметризуется Transformer и обучается восстанавливать замаскированные токены, оптимизируя нижнюю оценку правдоподобия. На генерации модель идёт от полностью замаскированной последовательности к полностью раскрытой, на каждом шаге предсказывая маски и при необходимости перемаскируя часть токенов. Обученная с нуля LLaDA на 8 миллиардов параметров сопоставима по качеству с LLaMA3 8B на широком наборе бенчмарков — это и есть главный аргумент работы: способности больших языковых моделей не обязаны опираться именно на авторегрессию.

LLaDA внутри использует обычный Transformer с вниманием — то есть по первому измерению она ничего не меняет. Нова именно вероятностная постановка: вместо «предсказать следующий токен слева направо» она моделирует распределение через прямой и обратный процессы маскирования, что даёт двунаправленный контекст и порождение всего ответа сразу с итеративным уточнением. У этого есть и обратная сторона: диффузионная выборка обычно требует многих шагов и пока дороже авторегрессионной, а длина контекста фиксируется наперёд (с этим борются приёмы вроде блочной диффузии и кэширования). Подход уже расширяют и на мультимодальность — например, LLaDA-V добавляет визуальный энкодер и обучение по визуальным инструкциям поверх той же диффузионной основы.
